In [1]:
#imports
from collections import  deque
import pygame
import random
import math

pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.7)


In [2]:

#   making the class
class particle():
    def __init__(self, x, y, weight, color):
        self.x = x
        self.y = y
        self.vx = 0
        self.vy = 0

        self.weight = weight
        self.color = color
        
        self.trail = deque(maxlen=20)

    def wall_collisions(self) :
        if p.x >= w_width - radius : 
            p.x =  w_width - radius
            p.vx *= -bounce

        if p.x <= radius:
            p.x = radius
            p.vx *= -bounce

        if p.y >= w_height - radius : 
            p.y =  w_height - radius
            p.vy *= -bounce

        if p.y <= radius:
            p.y = radius
            p.vy *= -bounce

In [3]:
# function to update the velocity 

def gravitate(b, dt):
    for i in range(len(b)):
        for j in range(i+1, len(b)):

            # getting distance of each particle
            p1 = b [i]
            p2 = b[j]

            dx = p2.x - p1.x 
            dy = p2.y - p1.y 
            distance = math.hypot(dx, dy)

            if distance == 0:
                continue
                # --- 1. COLLISION RESOLUTION ---
            if distance <= radius * 2:
                # Normal vector pointing from p1 to p2
                nx = dx / distance
                ny = dy / distance

                # Push them apart equally so they don't stick
                overlap = (radius * 2) - distance
                p1.x -= nx * overlap * 0.5
                p1.y -= ny * overlap * 0.5
                p2.x += nx * overlap * 0.5
                p2.y += ny * overlap * 0.5

                # Relative velocity between the two particles
                vx_rel = p1.vx - p2.vx
                vy_rel = p1.vy - p2.vy

                # Velocity along the normal vector
                vel_along_normal = vx_rel * nx + vy_rel * ny

                impulse = -(1 + bounce) * vel_along_normal / ((1 / p1.weight) + (1 / p2.weight))
                # Only resolve if they are moving toward each other
                if vel_along_normal > 0:
                    # Impulse scalar accounting for mass (weight)
                    impulse = -(1 + bounce) * vel_along_normal / ((1 / p1.weight) + (1 / p2.weight))

                    # Apply impulse to both velocities based on mass
                    p1.vx += (impulse / p1.weight) * nx
                    p1.vy += (impulse / p1.weight) * ny
                    p2.vx -= (impulse / p2.weight) * nx
                    p2.vy -= (impulse / p2.weight) * ny
                
                continue  # Skip gravity pull if they are actively colliding
            # the calculations formula for 11 particle pull strength
            # is G * M / R**2  
            accel1 = (G * p2.weight) / distance**2
            accel2 = (G * p1.weight) / distance**2

            p1.vx += accel1 * (dx/distance) * dt
            p1.vy += accel1 * (dy/distance) * dt     

            p2.vx -= accel2 * (dx/distance) * dt
            p2.vy -= accel2 * (dy/distance) * dt     


In [ ]:
#      initializing the screen
radius = 20
bounce = 1
color_list= [(220, 101, 220), (20, 110, 202), (0, 220, 100) ]

pygame.init()
w_height = 600
w_width = 800
G = 6000

screen = pygame.display.set_mode((w_width, w_height))
clock = pygame.time.Clock()


running = True
ball_count = 3
particles = []


for i in range(ball_count):
    x = random.randint(radius, w_width)
    y = random.randint(radius, w_height)
    weight = random.randint(200, 1000)
    color = color_list[i]
    
    particles.append(particle(x, y, weight, color))


while running:
    dt = clock.tick(60) /1000
    

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    
    gravitate(particles, dt)

    for p in particles:
        p.x += p.vx *dt
        p.y += p.vy *dt

        p.wall_collisions()

    screen.fill((0,0,0))
# Draw trails and particles
    for p in particles:
        # Draw the trail points (older points shrink and fade out)
        for index, pos in enumerate(p.trail):
            # Scale radius down for older parts of the trail
            trail_radius = int(p.radius * (index / len(p.trail)))
            if trail_radius > 0:
                # Dim the color based on how old the history point is
                fade_color = (
                    int(p.color[0] * (index / len(p.trail))),
                    int(p.color[1] * (index / len(p.trail))),
                    int(p.color[2] * (index / len(p.trail)))
                )
                pygame.draw.circle(screen, fade_color, (int(pos[0]), int(pos[1])), trail_radius)

        # Draw the main particle on top
        pygame.draw.circle(screen, p.color, (int(p.x), int(p.y)), radius)
    pygame.display.flip()

pygame.quit()

